# Two-layer neural network

University of Michigan EECS 498/598, Winter 2022. Compact companion to the completed course assignment.

**Static validation only; not executed.** No ordinary-notebook outputs or scores are copied here. Run cells in order only when execution is intended; training variants retain the ordinary notebook parameters. Existing course dependencies and data are required; no installation or repository cloning is performed. Results are isolated under `core_results/two_layer_net`.

## Setup

In [ ]:
from pathlib import Path
import os
import sys
import importlib

candidates = [Path.cwd(), *Path.cwd().parents, Path('/content/drive/MyDrive/umich-dlcv-wi2022')]
PROJECT_ROOT = next((p for p in candidates if (p / 'assignments' / 'a2' / 'two_layer_net_core.ipynb').is_file()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError('Open the canonical project or assignment directory; mount Drive first in Colab.')
ASSIGNMENT_DIR = PROJECT_ROOT / 'assignments' / 'a2'
if 'eecs598' in sys.modules and Path(sys.modules['eecs598'].__file__).resolve().parent != (ASSIGNMENT_DIR / 'eecs598').resolve():
    raise RuntimeError('Restart the kernel before switching assignments: eecs598 differs across assignments.')
os.chdir(ASSIGNMENT_DIR)
sys.path.insert(0, str(ASSIGNMENT_DIR))
sys.modules.setdefault('imp', importlib)
import torch
import matplotlib.pyplot as plt
import eecs598
from eecs598 import reset_seed

device = 'cuda' if torch.cuda.is_available() else 'cpu'
RESULTS_DIR = ASSIGNMENT_DIR / 'core_results' / 'two_layer_net'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
reset_seed(0)
print('Device:', device)


## Data

In [ ]:
DATA_DIR = next((p for p in (ASSIGNMENT_DIR, PROJECT_ROOT / 'assignments' / 'a1') if (p / 'cifar-10-batches-py').is_dir()), None)
if DATA_DIR is None:
    raise FileNotFoundError('Prepare the course CIFAR-10 cache before running this notebook.')
try:
    os.chdir(DATA_DIR)
    data_dict = eecs598.data.preprocess_cifar10(cuda=device == 'cuda', dtype=torch.float32, flatten=True, bias_trick=False, show_examples=False)
finally:
    os.chdir(ASSIGNMENT_DIR)


## Baseline and required hyperparameter comparisons

In [ ]:
from two_layer_net_core import TwoLayerNet, find_best_net, nn_get_search_params
def fit_network(hidden, lr, reg, iterations, batch_size):
    reset_seed(0)
    net = TwoLayerNet(3072, hidden, 10, dtype=data_dict['X_train'].dtype, device=device)
    stats = net.train(data_dict['X_train'], data_dict['y_train'], data_dict['X_val'], data_dict['y_val'], num_iters=iterations, batch_size=batch_size, learning_rate=lr, learning_rate_decay=0.95, reg=reg)
    return net, stats

baseline, baseline_stats = fit_network(36, 1e-2, 0.25, 500, 1000)
variants = {
    'width': [(h, 0.1, 0.001) for h in [2, 8, 32, 128]],
    'regularization': [(128, 1.0, r) for r in [0, 1e-5, 1e-3, 1e-1]],
    'learning_rate': [(128, lr, 1e-4) for lr in [1e-4, 1e-2, 1, 100]],
}
variant_stats = {}
for name, settings in variants.items():
    variant_stats[name] = {}
    for hidden, lr, reg in settings:
        net, stats = fit_network(hidden, lr, reg, 3000, 1000)
        variant_stats[name][(hidden, lr, reg)] = stats
        plt.plot(stats['val_acc_history'], label=f'h={hidden}, lr={lr}, reg={reg}')
    plt.title(name)
    plt.xlabel('Epoch')
    plt.ylabel('Validation accuracy')
    plt.legend()
    plt.savefig(RESULTS_DIR / f'{name}.png', bbox_inches='tight')
    plt.show()


## Train the validation-selected network

In [ ]:
reset_seed(0)
best_net, best_stats, best_val_acc = find_best_net(data_dict, nn_get_search_params)
best_net.save(str(RESULTS_DIR / 'nn_best_model.pt'))
torch.save({'baseline': baseline_stats, 'variants': variant_stats, 'best': best_stats, 'validation_accuracy': best_val_acc}, RESULTS_DIR / 'training_stats.pt')
print('Best validation accuracy:', best_val_acc)


## Evaluate

In [ ]:
print('Test accuracy:', (best_net.predict(data_dict['X_test']) == data_dict['y_test']).float().mean().item())
plt.plot(best_stats['loss_history'])
plt.xlabel('Iteration')
plt.ylabel('Training loss')
plt.savefig(RESULTS_DIR / 'best_loss.png', bbox_inches='tight')
plt.show()
